In [69]:
import pandas as pd
import numpy as np
from pathlib import Path

In [95]:
df=pd.read_csv("C:\\umyma2\\DeepFake-Detection\\processed-frames\\metadata.csv")

In [96]:
df.head()

,image_path,label,manipulation_type,compression,original_path
0,C:\umyma2\DeepFake-Detection\processed-frames\...,0,real,c23,C:\umyma2\DeepFake-Detection\frames\real\000\f...
1,C:\umyma2\DeepFake-Detection\processed-frames\...,0,real,c23,C:\umyma2\DeepFake-Detection\frames\real\000\f...
2,C:\umyma2\DeepFake-Detection\processed-frames\...,0,real,c23,C:\umyma2\DeepFake-Detection\frames\real\000\f...
3,C:\umyma2\DeepFake-Detection\processed-frames\...,0,real,c23,C:\umyma2\DeepFake-Detection\frames\real\000\f...
4,C:\umyma2\DeepFake-Detection\processed-frames\...,0,real,c23,C:\umyma2\DeepFake-Detection\frames\real\000\f...


In [ ]:
def get_vid_path(image_path): 
    filename=Path(image_path).stem     
    video_id=filename.split("_")[0]     
    return video_id

def split_vid_ids(video_ids):
    vid_ids=np.array(list(video_ids))
    np.random.seed(42)
    np.random.shuffle(vid_ids)
    total=len(vid_ids)
    train_end=int(total*0.70)
    val_end=int(total*0.85)
    train=set(vid_ids[:train_end])
    val=set(vid_ids[train_end:val_end])
    test=set(vid_ids[val_end:])
    return train,val,test

In [98]:
df["video_id"]=df["image_path"].apply(get_vid_path)

In [109]:
df["video_id"].head()

0    000
1    000
2    000
3    000
4    000
Name: video_id, dtype: str

In [99]:
all_train, all_test, all_val= set(),set(),set()
# initializing sets since each vid id is unique

In [100]:
def assign_split(video_id):
    if video_id in all_train:
        return "train"
    elif video_id in all_val:
        return "val"
    elif video_id in all_test:
        return "test"
    return "train"

In [101]:
for manip in df["manipulation_type"].unique():
    subset=df[df["manipulation_type"]==manip]
    vid_ids=subset["video_id"].unique()
    train_ids,val_ids,test_ids=split_vid_ids(vid_ids)
    all_train.update(train_ids)
    all_val.update(val_ids)
    all_test.update(test_ids)

In [ ]:
df["split"]=df["video_id"].apply(assign_split)

In [110]:
print(f"Train/Val overlap:  {len(train_ids & val_ids)}")
print(f"Train/Test overlap: {len(train_ids & test_ids)}")
print(f"Val/Test overlap:   {len(val_ids & test_ids)}")

Train/Val overlap:  0
Train/Test overlap: 0
Val/Test overlap:   0


In [111]:
print(f"Image distribution: {df.groupby(["split", "manipulation_type"]).size()}")

Image distribution: split  manipulation_type
test   Deepfakes             577
       Face2Face             605
       FaceSwap              462
       NeuralTextures        462
       real                  578
train  Deepfakes            3024
       Face2Face            2808
       FaceSwap             2281
       NeuralTextures       2281
       real                 3026
val    Deepfakes             631
       Face2Face             541
       FaceSwap              473
       NeuralTextures        473
       real                  631
dtype: int64


In [106]:
df.to_csv("C:\\umyma2\\DeepFake-Detection\\processed-frames\\metadata_splits.csv")